# 10 — Pilot-based GPU time and resource planner

**Purpose:** Estimate quota needs from REAL pilot timings; report assumptions instead of pretending estimates were observed full-run durations.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# 0. CLONE + ENVIRONMENT SETUP — RUN FIRST (Kaggle/Colab).
# Internet must be ON. The FIRST run may install libraries and STOP for a kernel restart.
# After restarting the kernel, rerun this cell; it will skip installation if versions match.
from pathlib import Path
import os, sys, subprocess, importlib.metadata as metadata

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
PROJECT_FOLDER = "Urdu_Qwen_Research_Notebooks"
PULL_LATEST = False  # Set True after pushing changes to GitHub; requires a clean Git checkout.

if Path('/kaggle/working').is_dir():
    WORKSPACE = Path('/kaggle/working')
elif Path('/content').is_dir():
    WORKSPACE = Path('/content')
else:
    WORKSPACE = Path.cwd()
REPO_DIR = WORKSPACE / 'nlp_research_proj'
ROOT = REPO_DIR / PROJECT_FOLDER

if (REPO_DIR / '.git').is_dir():
    if PULL_LATEST:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    if REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} exists but is not a Git clone. Rename/remove it after backing up outputs.')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
if not (ROOT / 'requirements.txt').is_file() or not (ROOT / 'project_utils.py').is_file():
    raise FileNotFoundError(f'Expected complete project at {ROOT}. Check the GitHub folder layout.')

# This environment was selected for a stable Transformers 4 training API.
if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError(f'Python {sys.version.split()[0]} not in tested support range 3.10–3.12.')
REQUIRED = {
    'transformers': '4.51.3',
    'peft': '0.15.2',
    'accelerate': '1.6.0',
    'datasets': '3.5.0',
    'tokenizers': '0.21.1',
    'huggingface-hub': '0.30.2',
    'safetensors': '0.5.3',
    'scikit-learn': '1.6.1',
    'pandas': '2.2.3',
    'numpy': '1.26.4',
    'matplotlib': '3.10.1',
    'psutil': '7.0.0',
    'ipywidgets': '8.1.6',
    'nbformat': '5.10.4',
    'pytest': '8.3.5',
}

def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

mismatch = {name: (installed_version(name), version)
            for name, version in REQUIRED.items()
            if installed_version(name) != version}
torchao = installed_version('torchao')
if mismatch or torchao is not None:
    print('Installing the project-locked environment; CUDA-enabled PyTorch is left untouched.')
    if mismatch:
        print('Packages to align:', mismatch)
    if torchao is not None:
        print('Removing optional incompatible torchao:', torchao)
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt'),
                    '--upgrade-strategy', 'only-if-needed'], check=True)
    raise RuntimeError('Installation finished. RESTART THE KAGGLE/COLAB KERNEL, then rerun this FIRST cell. '
                       'Do not train before restarting.')

# No package installation needed: import the shared research code only now.
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()

# Catch future API drift before downloading a 0.5B model.
import inspect
from transformers import TrainingArguments, Trainer
sig = inspect.signature(TrainingArguments).parameters
needed = ['output_dir', 'group_by_length', 'warmup_ratio', 'eval_strategy']
missing = [name for name in needed if name not in sig]
if missing:
    raise RuntimeError(f'Transformers does not match our 4.51.3 training API. Missing: {missing}')
api = check_training_api()
print('Ready. Project:', ROOT)
print('Transformers:', api['transformers'], '| PEFT:', api['peft'])
print('Trainer token argument:', api['trainer_tokenizer_key'])
print('No installation needed; proceed to the next cell.')


Full fine-tuning cannot be assigned one universal duration. Batch size, GPU model, token length, optimizer, and startup overhead matter. **No timing value below is a published benchmark until your own pilot runs.**

In [ ]:
import math, pandas as pd
pilots=[r for r in read_jsonl(ROOT/'outputs/metrics/train_runs.jsonl') if r.get('pilot')]
if not pilots:
    print('No pilot measurements yet. Run GPU pilot in notebook 00 first.')
else:
    out=[]
    for r in pilots:
        for examples_per_class in [10,25,50]:
            n=7*examples_per_class
            effective_batch=cfg()['per_device_batch_size']*cfg()['gradient_accumulation']
            estimated_steps=math.ceil(n/effective_batch)*cfg()['train_epochs']
            # Estimate from completed actual training timings. Warmup bias is possible.
            sps=r['wall_seconds']/max(r['global_steps'],1)
            out.append(dict(pilot=r['run_id'], method=r['method'],per_class=examples_per_class,
                            approx_optimizer_steps=estimated_steps,naive_minutes=round(sps*estimated_steps/60,1),
                            warmup_included=True,estimated_from_pilot_steps=r['global_steps']))
    display(pd.DataFrame(out))
    print('Use 10–20 steps for a steadier speed measurement; 2-step pilots include heavy warmup.')

### Practical quota and checkpoint advice

- For 2–3-epoch runs, start at 10/class and move to 25/class only after both methods work.
- Save checkpoints after **each finished run** to persistent storage (Google Drive on Colab, Kaggle dataset/output).
- A free notebook session can disconnect; saved files in ephemeral workspace may disappear.
- If you need to preserve checkpoints through interruptions, implement step-based checkpoints and resume in a dedicated long-run notebook. This project currently defaults to saving only completed runs to reduce storage.
- A 2-step pilot alone does not measure model download time or full training duration reliably.
- Measure `nvidia-smi` and peak `torch.cuda.max_memory_allocated()`; the latter is allocated memory, not full driver-level device utilization.